# Setup

## Environment

In [1]:
# %%bash
# 1. Parameterize target branch name
BRANCH="LLM"
MODEL = "Gemini"

# 2. Reset workspace directory for a clean slate
!rm -rf FoSpy

# 3. Fast clone single branch
!git clone --single-branch --branch $BRANCH https://github.com/errthumt/FoSpy.git FoSpy

# 4. Change notebook working directory to target folder
%cd FoSpy/LLM/sandbox
!git pull

# 5. Fast dependency installation using uv without touching native PyTorch
%pip install -q uv
!uv pip install -r colab_reqs.txt #--no-deps
#!uv pip install -q safetensors fsspec huggingface_hub


Cloning into 'FoSpy'...
remote: Enumerating objects: 5203, done.
remote: Counting objects: 100% (316/316), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 5203 (delta 229), reused 274 (delta 208), pack-reused 4887 (from 1)
Receiving objects: 100% (5203/5203), 81.95 MiB | 20.83 MiB/s, done.
Resolving deltas: 100% (2930/2930), done.
/content/FoSpy/LLM/sandbox
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 16.6 MB/s eta 0:00:00:00:0100:01
Using Python 3.12.13 environment at: /usr
Resolved 53 packages in 1.21s                                        
Prepared 4 packages in 2.41s                                             
Uninstalled 2 packages in 673ms
Installed 4 packages in 97ms                                
 + bitsandbytes==0.49.2
 + evaluate==0.4.6
 - huggingface-hub==1.19.0
 + huggingface-hub==0.36.2
 - transformers==5.12.0
 + transformers==4.57.6


## Python Imports

In [2]:
import json
import os
import re
from pathlib import Path
import google.genai as genai
from google.genai import types
from google.colab import userdata
from pprint import pp
from Gemini._utils import _get_key, _get_key_from_upload

## Globals and Helpers

In [3]:
# if needed, invoke asyncio for possible ui for secrets.
# %gui asyncio

INPUT_SOURCES = {
    # Comment out after successful prompt to avoid excess key calls
    # "Ba2Zn5As6 Supplemental Information": (
    #     "Ba2Zn5As6_SI.txt",
    #     "Solid-State Material Science",
    # )
}

# Define default path pointers based on your setup variables
MODEL = "Gemini"
INPUT_DIR = Path("inputs")
OUTPUT_FILE = Path(MODEL) / "output" / "tokens.json"

# # Configure the API connection safely using your Colab Secrets vault
# try:
#     GOOGLE_API_KEY = userdata.get('FoSpy_Testing_API_key')
# except Exception:
#     print("Could not get key from Colab secrets. Looking for key at secrets/FoSpy_Testing_API_key.txt...")
#     try:
#         target_path = Path("secrets/FoSpy_Testing_API_key.txt")
#         if not target_path.exists():
#             target_path.parent.mkdir(parents=True, exist_ok=True)
#             print("Could not find key file. Paste your API key below:")

#             target_path.write_bytes(input("API key: ").encode("utf-8"))

#             print("Uploaded key file to runtime.")
        
#         GOOGLE_API_KEY = target_path.read_text()

#     except Exception as e:
#         raise Exception(f"Failed to find or upload key. Exception: {e}")

#     print("Successfully found key.")



def get_prompt(input_text, context="None Specified"):
    return f"""
    Extract the fundamental scientific information pertaining to the synthetic
    method and product characterization described in the following input. Return
    the result as a JSON-formatted list of atomic, order-independent scientific information
    tokens. Each token must represent one fact, relationship or entity. Whenever
    possible, token values should be coercible to primitive data types,
    including separating quantities into a `value` and `units` fields. If
    specified, context refers to the scientific field or chemical category
    pertaining to the intended synthetic products.

    Rules:
    - Remove redundancy.
    - Normalize chemical formulas.
    - Normalize references to individual elements (outside chemical formulas) to their full element name.
    - Each token must be independent of sentence order.
    - Token meaning must be independent of narrative context.
    - A preamble before JSON output is optional, but must not contain any JSON-signaling characters such as {{ or [.
    - No further response characters are allowed after the JSON output.
    - Context input is to assist with your understanding of input, and should not be included in the output.

    Input:
    \"\"\"
    {input_text}
    \"\"\"

    Context:
    \"\"\"
    {context}
    \"\"\"
    """
upload= False
GOOGLE_API_KEY = _get_key("FoSpy_Testing_API_key")
upload = GOOGLE_API_KEY == "upload"

if upload:
    import asyncio
    from ipywidgets import FileUpload
    from IPython.display import display
    print("Upload secrets.json to widget below:")
    upload_widget = FileUpload(accept=".json", multiple=False)
    display(upload_widget)


Looking for API key with variable name 'FoSpy_Testing_API_key'.

Looking for os.environ['FoSpy_Testing_API_key']...
Could not find API key through environment variable.

Looking in Colab secrets with google.colab.userdata.get('FoSpy_Testing_API_key')...
Could not get API key through Colab secrets. Exception:
Requesting secret FoSpy_Testing_API_key timed out. Secrets can only be fetched when running from the Colab UI.

Looking for cached secrets at '/content/FoSpy/LLM/sandbox/Gemini/secrets.json'...
Could not find secrets. Delegating to secrets.json upload.
Upload secrets.json to widget below:


FileUpload(value={}, accept='.json', description='Upload')

In [4]:
if upload:
    await asyncio.sleep(0.1)
    input("Press enter to continue after uploading secrets")

In [10]:
if upload:
    GOOGLE_API_KEY = _get_key_from_upload("FoSpy_Testing_API_key", upload_widget)

GIT_PAT = _get_key("FoSpy_git_PAT", fallback=False)


Looking for 'FoSpy_Testing_API_key' key in cached secrets...


Exception: Could not get API key through cached secrets. Exception: 'FoSpy_Testing_API_key'

In [ ]:
fp = Path("Gemini/secrets.json")
if fp.exists():
    print("exists")

with fp.open("r") as f:
    secrets = json.load(f)

print(secrets['FoSpy_Testing_API_Key'])
GIT_PAT = _get_key("FoSpy_git_PAT", fallback=False)
print(GIT_PAT)

GOOGLE_API_KEY = _get_key("FoSpy_Testing_API_key")

# Get Model

## Build Model

In [ ]:


print("Authenticating google.genai with key...")
try:
    client = genai.Client(api_key=GOOGLE_API_KEY)
except Exception as e:
    raise Exception(f"Failed to authenticate with key. Exception: {e}")
print("Client connection authenticated and ready!")

# Main Logic

In [ ]:
# Ensure the workspace target directory physically exists
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

out = {}

for name, (file, context) in INPUT_SOURCES.items():
    nm_out = {}

    out[name] = nm_out

    target_file = INPUT_DIR / file
    if not target_file.exists():
        print(f"ERROR: Target input file missing at path: {target_file}")
        continue

    input_text = target_file.read_text()
    nm_out["input_text"] = input_text
    prompt = get_prompt(input_text, context)

    print(f"Sending text generation request for: {name}")
    # Force Gemini to strictly output structured schema data natively
    response = model.generate_content(
        prompt,
        generation_config={"response_mime_type": "application/json"}
    )



    response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
            )
        )

    
    raw = response.text
    nm_out["full response"] = raw

    # Validate the JSON output structure
    matched = re.search(r"[\{\[].*[\}\]]", raw, flags=re.DOTALL)
    if not matched:
        print(f"ERROR: No JSON array or object pattern isolated for {name}")
        continue

    json_str = matched.group(0)
    try:
        load = json.loads(json_str)
        nm_out["token list"] = load
        print(f"Successfully extracted tokens for: {name}")
        pp(nm_out)
    except json.JSONDecodeError:
        print(f"ERROR: Truncated or invalid JSON block isolated: {json_str}")

if OUTPUT_FILE.exists():
    with OUTPUT_FILE.open("r") as f:
        new_out = json.load(f)
    new_out.update(out)
    out = new_out

with OUTPUT_FILE.open("w") as f:
    json.dump(out, f, indent=4)

print(f"Pipeline executed successfully. Extraction matrix written to: {OUTPUT_FILE}")


In [ ]:
if GIT_PAT is not None and input("Push to GitHub? (y/n) ").lower() == "y":
    print("Pushing to GitHub...")

    !git remote set-url origin https://$GIT_PAT@github.com/errthumt/FoSpy.git
    !git add $OUTPUT_FILE
    !git commit -m "Update $OUTPUT_FILE"
    !git push